# Data preparation

## Running the Propp pipeline for every book in La comédie humaine

In [1]:
def process_book(book_path, spacy_model, mentions_detection_model, coreference_resolution_model):

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file(book_path)

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
      )

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

In [ ]:
import os
import gc
import datetime
import traceback
import torch
from propp_fr import load_models, save_tokens_df, save_entities_df, save_book_file

# Загружаем модели один раз для всех книг
spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

for subdir, dirs, files in os.walk("la_comedie_humaine"):
    for file in files:
        if not file.endswith(".txt"):
            continue

        book_name = file[:-4]
        book_path = os.path.join(subdir, file)

        # Пропускаем уже обработанные файлы
        book_file_path = os.path.join(subdir, book_name + "_book_file.book")
        if os.path.exists(book_file_path):
            print(f"Skipping (already processed): {file}")
            continue

        print("####################################################################################")
        print(f"Processing file: {file}")
        print(f"Start time: {datetime.datetime.now()}")

        try:
            tokens, entities, characters = process_book(
                book_path, spacy_model, mentions_detection_model, coreference_resolution_model
            )
            save_tokens_df(tokens, book_name + "_tokens", subdir)
            save_entities_df(entities, book_name + "_entities", subdir)
            save_book_file(characters, book_name + "_book_file", subdir)
            print(f"End time: {datetime.datetime.now()}")
        except Exception as e:
            print(f"ERROR processing {file}: {e}")
            traceback.print_exc()
        finally:
            # Очищаем память после каждой книги (и после ошибок тоже)
            gc.collect()
            torch.cuda.empty_cache()

C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


propp_fr package loaded successfully.
Loading models...
CUDA is required, Spacy model should run on GPU.
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH/final_model.pkl
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER/final_model

Models Loaded Successfully:
Spacy: fr_dep_news_trf
Mentions Detection: AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH
Coreference Resolution: AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER
Skipping (already processed): balzac-84-FC-physiologie-mariage.txt
Skipping (already processed): balzac-85-FC-petites-miseres-vie-conjugale.txt
Skipping (already processed): balzac-86-FC-pathologie-vie-sociale.txt
Skipping (already processed): balzac-86a-FC-traite-vie-elegante.txt
Skipping (already p

Batch Spacy Tokenization:   0%|          | 0/8 [00:00<?, ?it/s]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  12%|█▎        | 1/8 [00:05<00:39,  5.70s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  25%|██▌       | 2/8 [00:09<00:26,  4.36s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release

Loaded encoder model: almanach/camembert-large


Token indices sequence length is longer than the specified maximum sequence length for this model (152035 > 512). Running this sequence through the model will result in indexing errors


...

C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\propp_fr\propp_fr_coreference_resolution_module.py:1137: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  mention_pairs_df["confidence"].fillna(1, inplace=True)


14864	Illegal Coreference [2896, 2933]
15130	Illegal Coreference [5189, 5210]
15463	Illegal Coreference [5439, 5652]
End time: 2026-04-11 11:11:49.992541
####################################################################################
Processing file: balzac-37-FC-cabinet-antiques.txt
Start time: 2026-04-11 11:11:50.244293


Batch Spacy Tokenization:   0%|          | 0/5 [00:00<?, ?it/s]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  20%|██        | 1/5 [00:03<00:12,  3.09s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  40%|████      | 2/5 [00:05<00:08,  2.90s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release

Loaded encoder model: almanach/camembert-large


Token indices sequence length is longer than the specified maximum sequence length for this model (74926 > 512). Running this sequence through the model will result in indexing errors
